In [1]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine

database_host = 'bbdb-prod-master.postgres.database.azure.com'
database_user = 'mayank'
database_password = 'mB4shSR3F'
database_database = 'bbc'
conn = psycopg2.connect(host=database_host,database=database_database, user=database_user, password=database_password)
engine = create_engine('postgresql://{0}:{1}@{2}:5432/{3}'.format(database_user, database_password, database_host, database_database), connect_args={'sslmode':'require'})


In [2]:
POSTGRESQL_PARAMS = {
  'username': "HPzg5vzqsmye9PvIygPtXf2SVZrk16oi",
  'pass': "8GCacTSXObYR6nUllbx9SdF1KyT3psJX",
  'host': "bbdb-prod-master.postgres.database.azure.com",
  'DB': "bbc"
}

In [3]:
from bet365_scraper import get_bet365_events, get_odds_data, prepare_teams, prepare_competitions, prepare_bet365_events
from common import check_mappings, get_sqlalchemy_connection, psql_upsert_copy

In [4]:
import sys
sys.path.insert(1, '../bbpred/scrapers/')
BETSAPI_TOKEN ='134092-UszRvY4J1FIwdu'
SPORT_ID = '8'

In [5]:
# Add the folder containing the module to sys.path
# module_path = "E:\\"
module_path = "C:/Users/Rob/Desktop/"

if module_path not in sys.path:
    sys.path.append(module_path)

# Now import the module
import database_functions as dbf

In [6]:
import os
import numpy as np
import pandas as pd
import requests
import json
from time import sleep
import datetime
import uuid


In [7]:
sports_mapping = {
    '1': 'football',
    '8': 'rugby_union',
}

In [8]:
def get_betsapi_event_odds(BETSAPI_TOKEN, betsapi_event_id):
    event_odds_link = 'https://api.b365api.com/v2/event/odds/summary?token={0}&event_id={1}'.format(BETSAPI_TOKEN, betsapi_event_id)
    print(event_odds_link)
#     https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=5282567
    try:
        content = requests.get(event_odds_link)
        incoming_data = json.loads(content.text)
    except Exception as e:
        print(e)
        incoming_data = {'results': []}
    return incoming_data

In [9]:
def get_fixtures(SPORT_ID, betsapi_token, fixtures_to_retrieve, date_to_retrieve = None):
    
    if fixtures_to_retrieve == 'upcoming':
        upcoming_events_link = f'https://api.b365api.com/v3/events/upcoming?sport_id={SPORT_ID}&token={betsapi_token}'
    elif fixtures_to_retrieve == 'historic':
        if date_to_retrieve is None:
            date_to_retrieve = str(datetime.datetime.utcnow().date())
        upcoming_events_link = f"https://api.b365api.com/v3/events/ended?sport_id={SPORT_ID}&token={betsapi_token}&day={str(date_to_retrieve)}"
    elif fixtures_to_retrieve == 'future':
        if date_to_retrieve is None:
            date_to_retrieve = str(datetime.datetime.utcnow().date())
        upcoming_events_link = f"https://api.b365api.com/v3/events/upcoming?sport_id={SPORT_ID}&token={betsapi_token}&day={str(date_to_retrieve)}"
    else:
        print('fixtures_to_retrieve is not covered')
        return None
    print(upcoming_events_link)
    
    
    content = requests.get(upcoming_events_link)
    all_upcoming_games_data = json.loads(content.text)
    all_upcoming_games_dict = all_upcoming_games_data.get('results',dict())
    
    if len(all_upcoming_games_dict) > 0:
        all_upcoming_games = pd.DataFrame(all_upcoming_games_dict)
        all_upcoming_games['competition_external_id'] = all_upcoming_games['league'].apply(lambda x: x['id'])
        all_upcoming_games['competition_external_name'] = all_upcoming_games['league'].apply(lambda x: x['name'])
        all_upcoming_games['home_team_external_id'] = all_upcoming_games['home'].apply(lambda x: x['id'])
        all_upcoming_games['home_team_external_name'] = all_upcoming_games['home'].apply(lambda x: x['name'])
        all_upcoming_games['away_team_external_id'] = all_upcoming_games['away'].apply(lambda x: x['id'])
        all_upcoming_games['away_team_external_name'] = all_upcoming_games['away'].apply(lambda x: x['name'])
        all_upcoming_games = all_upcoming_games.drop(['league', 'home', 'away'], axis=1)
        all_upcoming_games['time'] = all_upcoming_games['time'].apply(lambda x: pd.to_datetime(datetime.datetime.utcfromtimestamp(int(x)), utc=True))
        all_upcoming_games['updated_at'] = datetime.datetime.utcnow()
        all_upcoming_games['source_id'] = '641913f8-1bf8-486d-841c-ad22d92368c5'
        all_upcoming_games['name'] = all_upcoming_games.apply(lambda x: x['home_team_external_name']+' vs '+x['away_team_external_name'], axis=1)
        all_upcoming_games['home_score'] = all_upcoming_games['ss'].apply(lambda x: np.nan if x is None else x.split('-')[0])
        all_upcoming_games['away_score'] = all_upcoming_games['ss'].apply(lambda x: np.nan if x is None else x.split('-')[1])
        all_upcoming_games['type'] = sports_mapping[SPORT_ID]
        all_upcoming_games = all_upcoming_games[~(all_upcoming_games['name'].str.contains(' 7s'))]
        all_upcoming_games = all_upcoming_games[~(all_upcoming_games['name'].str.contains(' Sevens'))]
        
    else:
        all_upcoming_games = pd.DataFrame()
    
    return all_upcoming_games

In [10]:
def get_summary_odds(event_ids):

    all_match_odds_dicts = []

    for betsapi_event_id in event_ids:

        odds_for_event = get_betsapi_event_odds(BETSAPI_TOKEN, betsapi_event_id)

        match_odds_dict = dict()
        match_odds_dict['event_id'] = betsapi_event_id


        odds_data = odds_for_event.get('results', None)
        
        if odds_data is not None:

            b365_data = odds_data.get('Bet365', None)

            if b365_data is not None:

                odds = b365_data.get('odds', None)

                if odds is not None:
                    

                    kick_off_odds = odds.get('kickoff', None)
                    
                    if kick_off_odds is not None:

                        match_odds = kick_off_odds['8_1']
                        if match_odds is not None:
                            match_odds_dict['home_odds_ko'] = match_odds['home_od']
                            match_odds_dict['draw_odds_ko'] = match_odds['draw_od']
                            match_odds_dict['away_odds_ko'] = match_odds['away_od']

                        match_odds = kick_off_odds['8_2']
                        if match_odds is not None:
                            match_odds_dict['handicap_odds_ko'] = match_odds['handicap']

                        match_odds = kick_off_odds['8_3']
                        if match_odds is not None:
                            match_odds_dict['total_points_odds_ko'] = match_odds['handicap']

                            
                            
                    start_odds = odds.get('start', None)
                    
                    if start_odds is not None:

                        match_odds = start_odds['8_1']
                        if match_odds is not None:
                            match_odds_dict['home_odds_start'] = match_odds['home_od']
                            match_odds_dict['draw_odds_start'] = match_odds['draw_od']
                            match_odds_dict['away_odds_start'] = match_odds['away_od']

                        match_odds = start_odds['8_2']
                        if match_odds is not None:
                            match_odds_dict['handicap_odds_start'] = match_odds['handicap']

                        match_odds = start_odds['8_3']
                        if match_odds is not None:
                            match_odds_dict['total_points_odds_start'] = match_odds['handicap']

                            
                    end_odds = odds.get('end', None)
                    
                    if end_odds is not None:

                        match_odds = end_odds['8_1']
                        if match_odds is not None:
                            match_odds_dict['home_odds_end'] = match_odds['home_od']
                            match_odds_dict['draw_odds_end'] = match_odds['draw_od']
                            match_odds_dict['away_odds_end'] = match_odds['away_od']

                        match_odds = end_odds['8_2']
                        if match_odds is not None:
                            match_odds_dict['handicap_odds_end'] = match_odds['handicap']

                        match_odds = end_odds['8_3']
                        if match_odds is not None:
                            match_odds_dict['total_points_odds_end'] = match_odds['handicap']

                        
                    all_match_odds_dicts.append(match_odds_dict)

    return pd.DataFrame(all_match_odds_dicts)

In [11]:
def update_event_ids():

    # Get the event_odds_b365 table
    sql_statement = "SELECT * from event_odds_b365;"
    event_odds_b365 = dbf.postgres_Retreive_Insert(sql_statement, POSTGRESQL_PARAMS, retrieve_data = True)

    # Get the event_ids for these events to match
    sql_statement = "SELECT es.event_id, es.external_event_id, tsc.team_id home_team_internal_id, tsc2.team_id away_team_internal_id from event_source es inner join team_source_comp tsc on es.home_team_external_id = tsc.external_id and es.competition_external_id = tsc.competition_external_id inner join team_source_comp tsc2 on es.away_team_external_id = tsc2.external_id and es.competition_external_id = tsc2.competition_external_id where es.source_id = '641913f8-1bf8-486d-841c-ad22d92368c5';"
    event_source = dbf.postgres_Retreive_Insert(sql_statement, POSTGRESQL_PARAMS, retrieve_data = True)

    # Merge the 2 data frames
    event_odds_b365 = event_odds_b365[['external_event_id', 'home_odds', 'away_odds', 'draw_odds', 'handicap', 'total_points', 'home_odds_ko', 'home_odds_end', 'home_odds_start', 'away_odds_ko', 'away_odds_end', 'away_odds_start', 'draw_odds_ko', 'draw_odds_end', 'draw_odds_start', 'handicap_odds_ko', 'handicap_odds_end', 'handicap_odds_start', 'total_points_odds_ko', 'total_points_odds_end', 'total_points_odds_start']].merge(event_source, how = 'left', left_on = 'external_event_id', right_on = 'external_event_id')

    # Check that the teams are roud the same way as we have them in order to pass the check
    sql_statement = "SELECT event_id, home_team_internal_id home_team_internal_id_ve, away_team_internal_id away_team_internal_id_ve from view_event;"
    view_event = dbf.postgres_Retreive_Insert(sql_statement, POSTGRESQL_PARAMS, retrieve_data = True)

    event_odds_b365 = event_odds_b365.merge(view_event, how = 'left', left_on = 'event_id', right_on = 'event_id')
    event_odds_b365['home_team_check'] = event_odds_b365['home_team_internal_id'] == event_odds_b365['home_team_internal_id_ve']
    event_odds_b365['away_team_check'] = event_odds_b365['away_team_internal_id'] == event_odds_b365['away_team_internal_id_ve']

    
    # Create the odds_columns to use
    event_odds_b365['home_odds'] = event_odds_b365[['home_odds_ko', 'home_odds_end', 'home_odds_start']].apply(lambda x: x[0] if pd.notna(x[0]) else x[1] if pd.notna(x[1]) else x[2], axis = 1)
    event_odds_b365['away_odds'] = event_odds_b365[['away_odds_ko', 'away_odds_end', 'away_odds_start']].apply(lambda x: x[0] if pd.notna(x[0]) else x[1] if pd.notna(x[1]) else x[2], axis = 1)
    event_odds_b365['draw_odds'] = event_odds_b365[['draw_odds_ko', 'draw_odds_end', 'draw_odds_start']].apply(lambda x: x[0] if pd.notna(x[0]) else x[1] if pd.notna(x[1]) else x[2], axis = 1)

    event_odds_b365['handicap'] = event_odds_b365[['handicap_odds_ko', 'handicap_odds_end', 'handicap_odds_start']].apply(lambda x: x[0] if pd.notna(x[0]) else x[1] if pd.notna(x[1]) else x[2], axis = 1)
    event_odds_b365['total_points'] = event_odds_b365[['total_points_odds_ko', 'total_points_odds_end', 'total_points_odds_start']].apply(lambda x: x[0] if pd.notna(x[0]) else x[1] if pd.notna(x[1]) else x[2], axis = 1)

    # Check the odds don't add up to something out of range to pass the check
    event_odds_b365['odds_probability'] = (1 / event_odds_b365['home_odds'].fillna(1000)) + (1 / event_odds_b365['draw_odds'].fillna(1000)) + (1 / event_odds_b365['away_odds'].fillna(1000))
    event_odds_b365['odds_check'] = (event_odds_b365['odds_probability'] <= 1.15) & (event_odds_b365['odds_probability'] >= 0.95)


    event_odds_b365['passed_check'] = (event_odds_b365['home_team_check'] == True) & (event_odds_b365['away_team_check'] == True) & (event_odds_b365['odds_check'] == True)

    cols_to_update = ['event_id', 'external_event_id', 'home_odds', 'away_odds', 'draw_odds', 'handicap', 'total_points', 'passed_check']
    dbf.add_data_to_database(event_odds_b365[cols_to_update], table_name = 'event_odds_b365', id_column = 'external_event_id', POSTGRESQL_PARAMS = POSTGRESQL_PARAMS)
    
    return


In [12]:
# We need to check that the teams are the same way we have them in the materialised view - If they are then we shouldd be able to crack on

In [13]:
# Use this to get historic events
# Define the start and end dates
start_date = datetime.datetime(2025, 1, 1)
# end_date = datetime.datetime(2023, 4, 1)
# start_date = datetime.datetime.now() - datetime.timedelta(days = 365*10)
end_date = datetime.datetime.now()

# Generate the list of dates
date_list = [(start_date + datetime.timedelta(days=i)).strftime('%Y-%m-%d') for i in range((end_date - start_date).days + 1)]

all_eventsData = pd.DataFrame()

for dl in date_list:
    
    print(dl)
    eventsData = get_fixtures(SPORT_ID, BETSAPI_TOKEN, fixtures_to_retrieve = 'historic', date_to_retrieve = dl)
    all_eventsData = pd.concat([all_eventsData, eventsData], ignore_index = True)
    
all_eventsData = all_eventsData[['id', 'sport_id', 'time', 'time_status', 'round', 'competition_external_id', 'competition_external_name',
       'home_team_external_id', 'home_team_external_name',
       'away_team_external_id', 'away_team_external_name',
       'source_id', 'name', 'home_score', 'away_score', 'type']].drop_duplicates()

2025-01-01
https://api.b365api.com/v3/events/ended?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-01-01
2025-01-02
https://api.b365api.com/v3/events/ended?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-01-02
2025-01-03
https://api.b365api.com/v3/events/ended?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-01-03
2025-01-04
https://api.b365api.com/v3/events/ended?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-01-04
2025-01-05
https://api.b365api.com/v3/events/ended?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-01-05
2025-01-06
https://api.b365api.com/v3/events/ended?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-01-06
2025-01-07
https://api.b365api.com/v3/events/ended?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-01-07
2025-01-08
https://api.b365api.com/v3/events/ended?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-01-08
2025-01-09
https://api.b365api.com/v3/events/ended?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-01-09
2025-01-10
https://api.b365api.com/v3/events/ended?spor

In [14]:
all_eventsData

,id,sport_id,time,time_status,round,competition_external_id,competition_external_name,home_team_external_id,home_team_external_name,away_team_external_id,away_team_external_name,source_id,name,home_score,away_score,type
0,8590746,8,2025-01-01 17:15:00+00:00,3,9,27321,United Rugby Championship,5933,Scarlets,47036,Newport Gwent Dragons,641913f8-1bf8-486d-841c-ad22d92368c5,Scarlets vs Newport Gwent Dragons,32,15,rugby_union
1,8590744,8,2025-01-01 15:00:00+00:00,3,9,27321,United Rugby Championship,6383,Cardiff Blues,29127,Ospreys,641913f8-1bf8-486d-841c-ad22d92368c5,Cardiff Blues vs Ospreys,13,13,rugby_union
2,8572621,8,2025-01-03 19:45:00+00:00,3,10,11970,Premiership Rugby,17141,Newcastle Falcons,6123,Harlequins,641913f8-1bf8-486d-841c-ad22d92368c5,Newcastle Falcons vs Harlequins,14,38,rugby_union
3,8574894,8,2025-01-04 20:05:00+00:00,3,14,596,French Rugby Championship,47042,La Rochelle,7351,Toulouse,641913f8-1bf8-486d-841c-ad22d92368c5,La Rochelle vs Toulouse,22,19,rugby_union
4,8571638,8,2025-01-04 17:30:00+00:00,3,10,11970,Premiership Rugby,5838,Saracens,6124,Bristol,641913f8-1bf8-486d-841c-ad22d92368c5,Saracens vs Bristol,35,26,rugby_union
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,9237089,8,2025-02-02 11:45:00+00:00,3,1,3746,Rugby Europe Championship,63873,Spain,479095,Netherlands,641913f8-1bf8-486d-841c-ad22d92368c5,Spain vs Netherlands,53,24,rugby_union
266,9342477,8,2025-02-02 11:00:00+00:00,3,11,1730,Georgia Premier Division,856041,RC Tao,188545,Aia Kutaisi,641913f8-1bf8-486d-841c-ad22d92368c5,RC Tao vs Aia Kutaisi,17,30,rugby_union
267,9454581,8,2025-02-02 07:00:00+00:00,3,NaN,39402,Japan Rugby League One - Div 2,325689,Green Rockets,68611,Kintetsu Liners,641913f8-1bf8-486d-841c-ad22d92368c5,Green Rockets vs Kintetsu Liners,19,31,rugby_union
268,9269831,8,2025-02-02 03:10:00+00:00,3,6,39428,Japan Rugby League One - Div 1,68587,Honda Heat,68728,Toshiba Brave Lupus,641913f8-1bf8-486d-841c-ad22d92368c5,Honda Heat vs Toshiba Brave Lupus,12,35,rugby_union


In [15]:
# Use this to get future events
# Define the start and end dates
start_date = datetime.datetime.now()
end_date = start_date + datetime.timedelta(days = 15)

# Generate the list of dates
date_list = [(start_date + datetime.timedelta(days=i)).strftime('%Y-%m-%d') for i in range((end_date - start_date).days + 1)]

future_eventsData = pd.DataFrame()

for dl in date_list:
    
    print(dl)
    eventsData = get_fixtures(SPORT_ID, BETSAPI_TOKEN, fixtures_to_retrieve = 'future', date_to_retrieve = dl)
    future_eventsData = pd.concat([future_eventsData, eventsData], ignore_index = True)
    
future_eventsData = future_eventsData[['id', 'sport_id', 'time', 'time_status', 'round', 'competition_external_id', 'competition_external_name',
       'home_team_external_id', 'home_team_external_name',
       'away_team_external_id', 'away_team_external_name',
       'source_id', 'name', 'home_score', 'away_score', 'type']].drop_duplicates()

2025-02-05
https://api.b365api.com/v3/events/upcoming?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-02-05
2025-02-06
https://api.b365api.com/v3/events/upcoming?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-02-06
2025-02-07
https://api.b365api.com/v3/events/upcoming?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-02-07
2025-02-08
https://api.b365api.com/v3/events/upcoming?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-02-08
2025-02-09
https://api.b365api.com/v3/events/upcoming?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-02-09
2025-02-10
https://api.b365api.com/v3/events/upcoming?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-02-10
2025-02-11
https://api.b365api.com/v3/events/upcoming?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-02-11
2025-02-12
https://api.b365api.com/v3/events/upcoming?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-02-12
2025-02-13
https://api.b365api.com/v3/events/upcoming?sport_id=8&token=134092-UszRvY4J1FIwdu&day=2025-02-13
2025-02-14
https://api.b365a

In [16]:
upcoming_eventsData = get_fixtures(SPORT_ID, BETSAPI_TOKEN, fixtures_to_retrieve = 'upcoming', date_to_retrieve = dl)
upcoming_eventsData = upcoming_eventsData[['id', 'sport_id', 'time', 'time_status', 'round', 'competition_external_id', 'competition_external_name',
       'home_team_external_id', 'home_team_external_name',
       'away_team_external_id', 'away_team_external_name',
       'source_id', 'name', 'home_score', 'away_score', 'type']].drop_duplicates()

https://api.b365api.com/v3/events/upcoming?sport_id=8&token=134092-UszRvY4J1FIwdu


In [17]:
all_eventsData = pd.concat([all_eventsData, future_eventsData, upcoming_eventsData], ignore_index = True).drop_duplicates()

In [18]:
eventsData = all_eventsData.copy()

In [15]:
# eventsData = get_fixtures(SPORT_ID, BETSAPI_TOKEN, fixtures_to_retrieve = 'upcoming')

In [19]:
competition_source_df = pd.read_sql_table('competition_source', schema='public', con=engine)
competition_source_df['source_id'] = competition_source_df['source_id'].apply(str)
competition_source_df['competition_id'] = competition_source_df['competition_id'].apply(str)

source_col = 'source_id'
id_col = 'external_id'
columns_to_check_for_changes = ['external_id', 'external_name']
internal_id_col = 'competition_id'
competitionsData = prepare_competitions(eventsData)
competitions_mapped = check_mappings(
    competitionsData, 
    competition_source_df, 
    source_col=source_col,
    id_col=id_col,
    columns_to_check_for_changes=columns_to_check_for_changes,
    internal_id_col=internal_id_col
)
engine = get_sqlalchemy_connection()

In [20]:
team_source_df = pd.read_sql_table('team_source_comp', schema='public', con=engine)
team_source_df['source_id'] = team_source_df['source_id'].apply(str)
team_source_df['team_id'] = team_source_df['team_id'].apply(str)

source_col = 'source_id'
id_col = ['external_id', 'competition_external_id']
columns_to_check_for_changes = ['external_id', 'competition_external_id', 'external_name']
internal_id_col = 'team_id'
teamsData = prepare_teams(eventsData)
teams_mapped = check_mappings(
    teamsData, 
    team_source_df, 
    source_col=source_col,
    id_col=id_col,
    columns_to_check_for_changes=columns_to_check_for_changes,
    internal_id_col=internal_id_col
)

In [21]:
eventsData = prepare_bet365_events(eventsData)


In [22]:
# Get the event_ids for these events to match
sql_statement = "SELECT id, external_event_id from event_source where source_id = '641913f8-1bf8-486d-841c-ad22d92368c5';"
event_source = dbf.postgres_Retreive_Insert(sql_statement, POSTGRESQL_PARAMS, retrieve_data = True)
event_source['exists'] = True
event_source.rename(columns = {'id':'current_id'}, inplace = True)

eventsData = eventsData.merge(event_source, how = 'left', left_on = ['external_event_id'], right_on = ['external_event_id'])
eventsData['id'] = eventsData[['current_id', 'id']].apply(lambda x: x[0] if pd.notna(x[0]) else x[1], axis = 1)
eventsData = eventsData.drop(['current_id', 'exists'], axis = 1)
dbf.add_data_to_database(eventsData, table_name = 'event_source', id_column = 'id', POSTGRESQL_PARAMS = POSTGRESQL_PARAMS)

Checking for events to update
                   start_time            start_time_old
0   2025-01-01 17:15:00+00:00 2025-01-01 17:15:00+00:00
1   2025-01-01 15:00:00+00:00 2025-01-01 15:00:00+00:00
2   2025-01-03 19:45:00+00:00 2025-01-03 19:45:00+00:00
3   2025-01-04 20:05:00+00:00 2025-01-04 20:05:00+00:00
4   2025-01-04 17:30:00+00:00 2025-01-04 17:30:00+00:00
..                        ...                       ...
401 2025-02-16 14:30:00+00:00 2025-02-16 14:30:00+00:00
402 2025-02-16 15:00:00+00:00 2025-02-16 15:00:00+00:00
403 2025-02-16 15:00:00+00:00 2025-02-16 15:00:00+00:00
404 2025-02-16 20:05:00+00:00 2025-02-15 16:00:00+00:00
405 2025-02-20 20:00:00+00:00 2025-02-21 16:00:00+00:00

[406 rows x 2 columns]
    competition_external_id competition_external_id_old
0                     27321                       27321
1                     27321                       27321
2                     11970                       11970
3                       596                       

(                   start_time external_event_id competition_external_id  \
 269 2025-02-05 14:27:00+00:00           9477170                   25464   
 314 2025-02-08 16:45:00+00:00           9461888                    3655   
 396 2025-02-15 17:15:00+00:00           8590758                   27321   
 408 2025-02-20 19:30:00+00:00           9389772                   38232   
 
     home_team_external_id away_team_external_id  \
 269                385451                385443   
 314                 63861                 63884   
 396                  5934                  5933   
 408                 68268                 51817   
 
                                 source_id  \
 269  641913f8-1bf8-486d-841c-ad22d92368c5   
 314  641913f8-1bf8-486d-841c-ad22d92368c5   
 396  641913f8-1bf8-486d-841c-ad22d92368c5   
 408  641913f8-1bf8-486d-841c-ad22d92368c5   
 
                                        name home_score away_score  \
 269  Credo-1963 Odessa vs Politehnik Odessa         2

In [24]:
# event_source_df = pd.read_sql_table('event_source', schema='public', con=engine)
# event_source_df['source_id'] = event_source_df['source_id'].apply(str)
# event_source_df['event_id'] = event_source_df['event_id'].apply(str)

# source_col = 'source_id'
# id_col = 'external_event_id'
# columns_to_check_for_changes = ['name', 'home_team_external_id', 'away_team_external_id', 'competition_external_id', 'start_time', 'home_score', 'away_score']
# internal_id_col = 'event_id'

# events_mapped = check_mappings(
#     eventsData, 
#     event_source_df, 
#     source_col=source_col,
#     id_col=id_col,
#     columns_to_check_for_changes=columns_to_check_for_changes,
#     internal_id_col=internal_id_col
# )

In [25]:
competitions_mapped.to_sql(name='competition_source', schema='public', con=engine, index=False, if_exists='append', method=psql_upsert_copy)
teams_mapped.to_sql(name='team_source_comp', schema='public', con=engine, index=False, if_exists='append', method=psql_upsert_copy)
# events_mapped.drop('event_id',axis = 1).to_sql(name='event_source', schema='public', con=engine, index=False, if_exists='append', method=psql_upsert_copy)

InvalidTextRepresentation: invalid input syntax for type uuid: "None"
CONTEXT:  COPY competition_source_staging, line 1, column competition_id: "None"


# Once events are mapped - Then get Odds

In [ ]:
eventsData[ eventsData['start_time'] >= '2025-01-26'][:20].sort_values('start_time')

In [23]:
# all_match_odds = pd.DataFrame()

# eventsData['date'] = eventsData['start_time'].apply(lambda x: x.date())
# eventsData.sort_values('date', ascending = True, inplace = True)
# dates_to_get = eventsData['date'].drop_duplicates()

# for dtg in dates_to_get:

#     print(dtg)
#     event_ids = list(eventsData[ eventsData['date'] == dtg]['external_event_id'] )

#     odds_for_date = get_summary_odds(event_ids)

#     if 'handicap' in odds_for_date.columns:
#         odds_for_date['handicap'] = odds_for_date['handicap'].apply(lambda x: x.replace('+',''))
    
#     if len(odds_for_date) > 0:
#         all_match_odds = pd.concat([all_match_odds, odds_for_date], ignore_index = True)
        
# all_match_odds.drop_duplicates('event_id', inplace = True)

In [24]:
import time

In [25]:
all_match_odds = pd.DataFrame()

eventsData['date'] = eventsData['start_time'].apply(lambda x: x.date())
eventsData.sort_values('date', ascending = True, inplace = True)
dates_to_get = eventsData['date'].drop_duplicates()

event_ids = list(eventsData['external_event_id'].drop_duplicates())
loop_num = 0

for event_id in event_ids:

    loop_num += 1
    print(loop_num, len(event_ids))
    
    odds_for_date = get_summary_odds([event_id])

    if 'handicap' in odds_for_date.columns:
        odds_for_date['handicap'] = odds_for_date['handicap'].apply(lambda x: x.replace('+',''))
    
    if len(odds_for_date) > 0:
        all_match_odds = pd.concat([all_match_odds, odds_for_date], ignore_index = True)
        
    if (loop_num % 2500) == 0:

        all_match_odds['handicap_odds_start'] = all_match_odds['handicap_odds_start'].apply(lambda x: str(x).replace('+',''))
        all_match_odds['handicap_odds_end'] = all_match_odds['handicap_odds_end'].apply(lambda x: str(x).replace('+',''))
        all_match_odds['handicap_odds_ko'] = all_match_odds['handicap_odds_ko'].apply(lambda x: str(x).replace('+',''))

        for col in ['home_odds_ko', 'away_odds_ko', 'draw_odds_ko', 'home_odds_start', 'away_odds_start', 'draw_odds_start', 'home_odds_end', 'away_odds_end', 'draw_odds_end']:
            all_match_odds[col] = all_match_odds[col].apply(lambda x: None if x == '-' else x)
    
        all_match_odds.drop_duplicates('event_id', inplace = True)

        odds_to_update = all_match_odds.rename(columns = {'event_id':'external_event_id'})
        cols_to_update = ['external_event_id', 'home_odds_ko', 'away_odds_ko', 'draw_odds_ko', 'handicap_odds_ko', 'total_points_odds_ko', 'home_odds_start', 'away_odds_start', 'draw_odds_start', 'handicap_odds_start', 'total_points_odds_start', 'home_odds_end', 'away_odds_end', 'draw_odds_end', 'handicap_odds_end', 'total_points_odds_end']
        # cols_to_update = ['external_event_id', 'home_odds_ko']
        for col in cols_to_update:
            if col not in odds_to_update.columns:
                print(col, 'not in the dataframe so adding')
                odds_to_update[col] = None


        dbf.add_data_to_database(odds_to_update[cols_to_update], table_name = 'event_odds_b365', id_column = 'external_event_id', POSTGRESQL_PARAMS = POSTGRESQL_PARAMS)


        update_event_ids()

        time.sleep(60 * 65)
        


1 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8590746
2 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8590744
3 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8572621
4 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=9269852
5 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=9269855
6 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=9269854
7 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=9269853
8 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8574893
9 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8651420
10 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8651421
11 410
https://api.b365api.co

85 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8666853
86 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=9363294
87 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8666765
88 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8667424
89 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8666767
90 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8600346
91 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8629158
92 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8629361
93 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8629360
94 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8629157
95 410
https://api.b

168 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8629365
169 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8629363
170 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8629364
171 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8629366
172 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8629899
173 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8629161
174 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8572625
175 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8626374
176 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8626436
177 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8629159
178 410
ht

251 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8709108
252 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8738260
253 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=9237098
254 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8699586
255 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=9195235
256 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8703020
257 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8702989
258 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8709088
259 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8703019
260 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=7982848
261 410
ht

334 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8702998
335 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8590755
336 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8590754
337 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8609903
338 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8629897
339 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8629373
340 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8629371
341 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8629166
342 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8608006
343 410
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8608004
344 410
ht

In [26]:
odds_to_update = all_match_odds.rename(columns = {'event_id':'external_event_id'})
cols_to_update = ['external_event_id', 'home_odds_ko', 'away_odds_ko', 'draw_odds_ko', 'handicap_odds_ko', 'total_points_odds_ko', 'home_odds_start', 'away_odds_start', 'draw_odds_start', 'handicap_odds_start', 'total_points_odds_start', 'home_odds_end', 'away_odds_end', 'draw_odds_end', 'handicap_odds_end', 'total_points_odds_end']
# cols_to_update = ['external_event_id', 'home_odds_ko']
for col in cols_to_update:
    if col not in odds_to_update.columns:
        print(col, 'not in the dataframe so adding')
        odds_to_update[col] = None

for col in ['home_odds_ko', 'away_odds_ko', 'draw_odds_ko', 'home_odds_start', 'away_odds_start', 'draw_odds_start', 'home_odds_end', 'away_odds_end', 'draw_odds_end']:
    odds_to_update[col] = odds_to_update[col].apply(lambda x: None if x == '-' else x)

odds_to_update.drop_duplicates('external_event_id', inplace = True)

odds_to_update['handicap_odds_start'] = odds_to_update['handicap_odds_start'].apply(lambda x: str(x).replace('+','') if pd.notna(x) else None)
odds_to_update['handicap_odds_end'] = odds_to_update['handicap_odds_end'].apply(lambda x: str(x).replace('+','') if pd.notna(x) else None)
odds_to_update['handicap_odds_ko'] = odds_to_update['handicap_odds_ko'].apply(lambda x: str(x).replace('+','') if pd.notna(x) else None)


dbf.add_data_to_database(odds_to_update[cols_to_update], table_name = 'event_odds_b365', id_column = 'external_event_id', POSTGRESQL_PARAMS = POSTGRESQL_PARAMS)


home_odds_ko not in the dataframe so adding
away_odds_ko not in the dataframe so adding
draw_odds_ko not in the dataframe so adding
handicap_odds_ko not in the dataframe so adding
total_points_odds_ko not in the dataframe so adding
Checking for events to update
Inserting new events: 17
insert_sql - Inserts Complete - 17
add_data_to_database - Complete


(    external_event_id home_odds_ko away_odds_ko draw_odds_ko handicap_odds_ko  \
 178           8629162         None         None         None             None   
 179           8629163         None         None         None             None   
 180           8629368         None         None         None             None   
 181           8629164         None         None         None             None   
 182           8629370         None         None         None             None   
 183           8629369         None         None         None             None   
 184           8629898         None         None         None             None   
 185           8629367         None         None         None             None   
 186           9461888         None         None         None             None   
 187           9269827         None         None         None             None   
 188           9269828         None         None         None             None   
 189           9

In [27]:
update_event_ids()

Checking for events to update
                                  event_id  \
7     5af8d279-ab37-11ef-a977-6045bdcfcd7b   
52    0e1a081e-9b05-11ec-98e5-0242ac140002   
114   a39895b9-d0d7-11ef-9202-6045bdcfcd7b   
118   33140552-112f-11ed-b46d-0242ac130002   
138   b73d2368-8cdf-11ef-8670-6045bdcfcd7b   
171   d59dbb38-0810-11ed-9f8a-0242ac130002   
683   daedab06-af21-11ea-81de-4865ee11b869   
684   9ba13992-3e7d-11ee-9cc9-0242ac120002   
701   221a13e4-6499-11ef-beb0-0242ac110002   
739   8415d60e-cd4d-11ef-979c-6045bdcfcd7b   
740   2244b4aa-6499-11ef-beb0-0242ac110002   
741   22a8e6dc-6499-11ef-beb0-0242ac110002   
742   8472fd2d-cd4d-11ef-957a-6045bdcfcd7b   
743   2276af6e-6499-11ef-beb0-0242ac110002   
744   848f996a-cd4d-11ef-92ad-6045bdcfcd7b   
745   21ef486c-6499-11ef-beb0-0242ac110002   
746                                   None   
747   85e52802-cd4d-11ef-9ab7-6045bdcfcd7b   
748   858efd50-cd4d-11ef-8f8a-6045bdcfcd7b   
749   85abac2e-cd4d-11ef-a952-6045bdcfcd7b   
750 

In [28]:
# Check if we have events in our source table that haven't been picked up in this run

In [29]:

# Get the event_odds_b365 table
sql_statement = "SELECT * from event_odds_b365;"
event_odds_b365 = dbf.postgres_Retreive_Insert(sql_statement, POSTGRESQL_PARAMS, retrieve_data = True)

# Get the event_ids for these events to match
sql_statement = "SELECT es.event_id, es.start_time, es.external_event_id, tsc.team_id home_team_internal_id, tsc2.team_id away_team_internal_id from event_source es inner join team_source_comp tsc on es.home_team_external_id = tsc.external_id and es.competition_external_id = tsc.competition_external_id inner join team_source_comp tsc2 on es.away_team_external_id = tsc2.external_id and es.competition_external_id = tsc2.competition_external_id where es.source_id = '641913f8-1bf8-486d-841c-ad22d92368c5';"
event_source = dbf.postgres_Retreive_Insert(sql_statement, POSTGRESQL_PARAMS, retrieve_data = True)


In [33]:
# event_source['start_time'] = pd.to_datetime(event_source['start_time'])
event_source["start_time"] = event_source["start_time"].dt.tz_localize(None)
event_source = event_source[ (event_source['start_time'] >= (datetime.datetime.now() - datetime.timedelta(days = 7))) & (event_source['start_time'] <= (datetime.datetime.now() + datetime.timedelta(days = 14)))]

In [34]:
eventsData = event_source.copy()

In [35]:
all_match_odds = pd.DataFrame()

eventsData['date'] = eventsData['start_time'].apply(lambda x: x.date())
eventsData.sort_values('date', ascending = False, inplace = True)
dates_to_get = eventsData['date'].drop_duplicates()

for dtg in dates_to_get:

    print(dtg)
    event_ids = list(eventsData[ eventsData['date'] == dtg]['external_event_id'] )

    odds_for_date = get_summary_odds(event_ids)

    if 'handicap' in odds_for_date.columns:
        odds_for_date['handicap'] = odds_for_date['handicap'].apply(lambda x: x.replace('+',''))
    
    if len(odds_for_date) > 0:
        all_match_odds = pd.concat([all_match_odds, odds_for_date], ignore_index = True)
        

2025-02-14
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=9236522
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=9236521
2025-02-13
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8629374
2025-02-09
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8600310
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8600359
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=9237091
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=9269826
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=9249646
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8709084
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8600358
https://api.b365api.com/v2/event/odds/summary?token=13409

https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8709088
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8709108
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8738260
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8703001
2025-01-31
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=9454430
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=9237088
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=9342478
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8709089
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=8738259
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1FIwdu&event_id=7982847
2025-01-30
https://api.b365api.com/v2/event/odds/summary?token=134092-UszRvY4J1

In [36]:
odds_to_update = all_match_odds.rename(columns = {'event_id':'external_event_id'})
cols_to_update = ['external_event_id', 'home_odds_ko', 'away_odds_ko', 'draw_odds_ko', 'handicap_odds_ko', 'total_points_odds_ko', 'home_odds_start', 'away_odds_start', 'draw_odds_start', 'handicap_odds_start', 'total_points_odds_start', 'home_odds_end', 'away_odds_end', 'draw_odds_end', 'handicap_odds_end', 'total_points_odds_end']
# cols_to_update = ['external_event_id', 'home_odds_ko']
for col in cols_to_update:
    if col not in odds_to_update.columns:
        print(col, 'not in the dataframe so adding')
        odds_to_update[col] = None

for col in ['home_odds_ko', 'away_odds_ko', 'draw_odds_ko', 'home_odds_start', 'away_odds_start', 'draw_odds_start', 'home_odds_end', 'away_odds_end', 'draw_odds_end']:
    odds_to_update[col] = odds_to_update[col].apply(lambda x: None if x == '-' else x)

odds_to_update.drop_duplicates('external_event_id', inplace = True)

odds_to_update['handicap_odds_start'] = odds_to_update['handicap_odds_start'].apply(lambda x: str(x).replace('+','') if pd.notna(x) else None)
odds_to_update['handicap_odds_end'] = odds_to_update['handicap_odds_end'].apply(lambda x: str(x).replace('+','') if pd.notna(x) else None)
odds_to_update['handicap_odds_ko'] = odds_to_update['handicap_odds_ko'].apply(lambda x: str(x).replace('+','') if pd.notna(x) else None)


dbf.add_data_to_database(odds_to_update[cols_to_update], table_name = 'event_odds_b365', id_column = 'external_event_id', POSTGRESQL_PARAMS = POSTGRESQL_PARAMS)


home_odds_ko not in the dataframe so adding
away_odds_ko not in the dataframe so adding
draw_odds_ko not in the dataframe so adding
handicap_odds_ko not in the dataframe so adding
total_points_odds_ko not in the dataframe so adding
Checking for events to update
add_data_to_database - Complete


(Empty DataFrame
 Columns: [external_event_id, home_odds_ko, away_odds_ko, draw_odds_ko, handicap_odds_ko, total_points_odds_ko, home_odds_start, away_odds_start, draw_odds_start, handicap_odds_start, total_points_odds_start, home_odds_end, away_odds_end, draw_odds_end, handicap_odds_end, total_points_odds_end]
 Index: [],
 Empty DataFrame
 Columns: [external_event_id, home_odds_ko, away_odds_ko, draw_odds_ko, handicap_odds_ko, total_points_odds_ko, home_odds_start, away_odds_start, draw_odds_start, handicap_odds_start, total_points_odds_start, home_odds_end, away_odds_end, draw_odds_end, handicap_odds_end, total_points_odds_end, home_odds_ko_old, away_odds_ko_old, draw_odds_ko_old, handicap_odds_ko_old, total_points_odds_ko_old, home_odds_start_old, away_odds_start_old, draw_odds_start_old, handicap_odds_start_old, total_points_odds_start_old, home_odds_end_old, away_odds_end_old, draw_odds_end_old, handicap_odds_end_old, total_points_odds_end_old, home_odds_ko_same, away_odds_ko_same

In [38]:
update_event_ids()

Checking for events to update
                                  event_id  \
8     5af8d279-ab37-11ef-a977-6045bdcfcd7b   
11    756aeeb4-cd4d-11ef-acd2-6045bdcfcd7b   
28    75e22932-cd4d-11ef-927f-6045bdcfcd7b   
52    0e1a081e-9b05-11ec-98e5-0242ac140002   
70    d92fb69e-aff7-11ee-b152-6045bdcf653a   
77    6c539dd8-dfc9-11ef-b359-6045bdcfcd7b   
87    cc8f8d6e-6498-11ef-beb0-0242ac110002   
93    6c8f3860-dfc9-11ef-836c-6045bdcfcd7b   
117   60c610b6-298b-11ed-87bb-0242ac130002   
2176  7a0bae7a-cd32-11ef-a206-6045bdcfcd7b   
4003  cc8f8d6e-6498-11ef-beb0-0242ac110002   
4640  9abafcbb-8c8b-11ef-8767-6045bdcfcd7b   
6068  d6c65cd0-211a-11ee-85de-0242ac120002   
6365  b73d2368-8cdf-11ef-8670-6045bdcfcd7b   
7275  90f8c7fa-093e-11ed-8c50-0242ac130002   
7605  b07a90c4-0ebe-11ed-8871-0242ac130002   

                              event_id_old  
8     5af8d279-ab37-11ef-a977-6045bdcfcd7b  
11                                    None  
28                                    None  
52    0